## Minh họa trực quan về cách mà **Vision Transformer** (ViT) hoạt động.
### *Là implement step-by-step trong blog về Vision Transformer của Quang-Huy Tran*.

In [224]:
import torch

### Thực hiện trên 1 batch = 1 sample. giả sử ảnh đầu vào dạng I là ảnh xám, size=(4, 4).
H = 4, W = 4, C = 1.

In [225]:
I = torch.arange(start=1, end=17).reshape(-1, 4)
I

tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12],
        [13, 14, 15, 16]])

### Tính toán số patch $N$, với mỗi patch có size là $P$. 
Giả sử $P=2$. số patches theo công thức:
$$
N = \frac{H \times W}{P^2} = \frac{4 \times 4}{4} = 4
$$

In [226]:
h, w = I.shape[0], I.shape[1]
patch_size = 2
num_patch = h*w//(patch_size**2)

print("number of patches:", num_patch)

number of patches: 4


### Lấy các patch từ $I$, flatten và xếp chồng.
Mỗi patch có kích thước $P \times P \times C$.  Sau khi flatten thì được vector chứa $P^2C$ chiều.  
Xếp Chồng các vector patch sau làm phẳng. Kết quả, tạo ra $X_{patch}$ có $N$ tokens, mỗi token gồm $P^2C$ chiều.
$$
X_{\text{patch}} \in \mathbb{R}^{N \times (P^2 \cdot C)}
$$

In [227]:
X_patch = []
for i_h in range(0, h, patch_size):
    for i_w in range(0, w, patch_size):
        patch = I[i_h : i_h + patch_size, i_w : i_w + patch_size]
        patch = patch.flatten().tolist()
        X_patch.append(patch)

X_patch = torch.tensor(X_patch, dtype=torch.float32)
print(X_patch)

tensor([[ 1.,  2.,  5.,  6.],
        [ 3.,  4.,  7.,  8.],
        [ 9., 10., 13., 14.],
        [11., 12., 15., 16.]])


### Patch Embedding: Flexible số chiều của các vector token (tương tự word embedding)
Ánh xạ $X_{patch}$ từ các vector $P^2C$ chiều sang không gian Vector Embedding gồm $D$ chiều.  
Trong bài này, ta sử dụng ánh xạ tuyến tính để chuyển từ (4, 4) về (4, 3), với bộ weight và bias như sau:
$$
W_E =
\begin{bmatrix}
0.1 & 0.2 & 0.3 \\
0.4 & 0.5 & 0.6 \\
0.7 & 0.8 & 0.9 \\
1.0 & 1.1 & 1.2
\end{bmatrix},
\quad
b_E = [0.1,\, 0.2,\, 0.3].
$$
Ta có công thức:
$$
E = X_{patch} \times W_E + b_E
$$
Lúc này, $E$ đã có dạng giống với ma trận sau embedding của bài toán text.

In [228]:
W_E = torch.tensor([
    [0.1, 0.2, 0.3],
    [0.4, 0.5, 0.6],
    [0.7, 0.8, 0.9],
    [1.0, 1.1, 1.2]
], dtype=torch.float32)
b_E = torch.tensor([0.1, 0.2, 0.3])
E = torch.matmul(X_patch, W_E) + b_E
print(E)

tensor([[10.5000, 12.0000, 13.5000],
        [14.9000, 17.2000, 19.5000],
        [28.1000, 32.8000, 37.5000],
        [32.5000, 38.0000, 43.5000]])


### Thêm token [CLS] vào đầu của ma trận (phần tử đầu tiên). 
Giả sử token CLS có dạng:
$$
x_{CLS} = [0.3,\, 0.5,\, 0.7]
$$
Nối $x_{CLS}$ vào đầu ma trận embedding $E$, ta được:

In [229]:
x_cls = torch.tensor([0.3, 0.5, 0.7]).reshape(1, -1)
E = torch.cat([x_cls, E], dim=0)
E

tensor([[ 0.3000,  0.5000,  0.7000],
        [10.5000, 12.0000, 13.5000],
        [14.9000, 17.2000, 19.5000],
        [28.1000, 32.8000, 37.5000],
        [32.5000, 38.0000, 43.5000]])

### Positional Encoding
Thêm thông tin về vị trí vào $E$. Giả sử ta đang có một positional encoding cho sẵn như sau:
$$
E_{\text{pos}} =
\begin{bmatrix}
0.0 & 0.0 & 0.0 \\
0.1 & 0.2 & 0.3 \\
0.4 & 0.5 & 0.6 \\
0.7 & 0.8 & 0.9 \\
1.0 & 1.1 & 1.2
\end{bmatrix}.
$$
Ta bổ sung thông tin về vị trí ở trên bằng cách cộng thẳng vào $E$:
$$
Z_0 = E + E_{pos}
$$


In [230]:
# khởi tạo ma trận E_pos
e_pos = torch.arange(0.1, 1.3, step=0.1).reshape(-1, 3)
x_zeros = torch.tensor([0.0, 0.0, 0.0]).reshape(1, -1)
e_pos = torch.cat([x_zeros, e_pos], dim=0)
e_pos

tensor([[0.0000, 0.0000, 0.0000],
        [0.1000, 0.2000, 0.3000],
        [0.4000, 0.5000, 0.6000],
        [0.7000, 0.8000, 0.9000],
        [1.0000, 1.1000, 1.2000]])

In [231]:
# Thêm thông tin vị trí vào E.
z_0 = E + e_pos
z_0

tensor([[ 0.3000,  0.5000,  0.7000],
        [10.6000, 12.2000, 13.8000],
        [15.3000, 17.7000, 20.1000],
        [28.8000, 33.6000, 38.4000],
        [33.5000, 39.1000, 44.7000]])

Ma trận $Z_0$ là đầu vào cuối cùng của Transformer Encoder.  
Hàng đầu tiên là [CLS] Token, các hàng sau là các patch ảnh đã được flatten cộng thêm thông tin về vị trí.

### **Transformer Encoder (PreLayerNorm)**
Quy trình thực hiện Transformer Encoder bao gồm các bước:
- Tính Layer Normalization với input là ma trận $Z_0$.
- self-attention, để tính toán đơn giản ta giả sử là single head.
- Skip connection.
- Layer Norm lần 2.
- Feed-forward (MLP).
- Skip connection lần 2.  
  
**Kì vọng đầu ra của Transformer Encoder là ma trận có shape giống với đầu vào.**

### Step 01: LayerNorm
*LayerNorm được áp dụng cho từng token trong batch, Trong ViT gọi là patch. Vậy ta áp dụng theo từng hàng.*  
Xét $i$ là chỉ số theo hàng, $j$ là chỉ số theo cột. Gọi $\mu_i$ là giá trị trung bình (mean) hàng $i$, $\sigma_i$ là độ lệch chuẩn (std) của  hàng $i$. Phần tử ban đầu ở hàng $i$ cột $j$ là $Z_0[i, j]$.  
  
Khi đó, công thức chuẩn hóa LayerNorm cho phần tử sẽ là:  
$$
Z'_0[i, j] = \frac{Z_0[i, j] - \mu_i}{\sigma_i}
$$

Kết quả, từ ma trận có dạng:
$$
Z_0 =
\begin{bmatrix}
0.3 & 0.5 & 0.7 \\
10.6 & 12.2 & 13.8 \\
15.3 & 17.7 & 20.1 \\
28.8 & 33.6 & 38.4 \\
33.5 & 39.1 & 44.7
\end{bmatrix}
\in \mathbb{R}^{5 \times 3}
$$
Sau khi chuẩn hóa sẽ được:
$$
Z'_0 =
\begin{bmatrix}
-1.2247 & 0 & 1.2247 \\
-1.2247 & 0 & 1.2247 \\
-1.2247 & 0 & 1.2247 \\
-1.2247 & 0 & 1.2247 \\
-1.2247 & 0 & 1.2247
\end{bmatrix}
\in \mathbb{R}^{5 \times 3}
$$

In [232]:
z_0 

tensor([[ 0.3000,  0.5000,  0.7000],
        [10.6000, 12.2000, 13.8000],
        [15.3000, 17.7000, 20.1000],
        [28.8000, 33.6000, 38.4000],
        [33.5000, 39.1000, 44.7000]])

In [233]:
z_0_mean = z_0.mean(dim=1).reshape(-1, 1)
z_0_std = z_0.std(dim=1, unbiased=False).reshape(-1, 1)

z_0_layernorm = (z_0 - z_0_mean)/z_0_std
torch.set_printoptions(precision=4, sci_mode=False)                # thay đổi cách hiển thị, không thay đổi giá trị tính bên trong
z_0_layernorm

tensor([[-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247]])

### Step 02. Self-attention (single-head)
Bước tính quan trọng nhất của Transformer Encoder. Đầu tiên, ta thiết lập các ma trận trọng số $W_Q, W_K, W_V$:
$$
W_Q =
\begin{bmatrix}
0.2 & 0.1 & 0.2 \\
0.3 & 0.4 & 0.3 \\
0.4 & 0.5 & 0.6
\end{bmatrix},
\quad
W_K =
\begin{bmatrix}
0.1 & 0.2 & 0.3 \\
0.4 & 0.5 & 0.6 \\
0.7 & 0.8 & 0.9
\end{bmatrix},
\quad
W_V =
\begin{bmatrix}
0.1 & 0.2 & 0.3 \\
0.4 & 0.5 & 0.6 \\
0.7 & 0.8 & 0.9
\end{bmatrix}.
$$

In [234]:
W_Q = torch.tensor([
    [0.2, 0.1, 0.2],
    [0.3, 0.4, 0.3],
    [0.4, 0.5, 0.6]
], dtype=torch.float32)
W_Q

tensor([[0.2000, 0.1000, 0.2000],
        [0.3000, 0.4000, 0.3000],
        [0.4000, 0.5000, 0.6000]])

In [235]:
W_K = W_V = torch.arange(0.1, 1.0, 0.1, dtype=torch.float32).reshape(-1, 3)
W_K, W_V

(tensor([[0.1000, 0.2000, 0.3000],
         [0.4000, 0.5000, 0.6000],
         [0.7000, 0.8000, 0.9000]]),
 tensor([[0.1000, 0.2000, 0.3000],
         [0.4000, 0.5000, 0.6000],
         [0.7000, 0.8000, 0.9000]]))

Sử dụng các ma trận trọng số trên nhân với $Z_0$ đã LayerNorm, ta thu được các ma trận $Q$ (Query), $K$ (key) và $V$ (Value).
$$
Q = Z'_0 \cdot W_Q \quad K = Z'_0 \cdot Q_K \quad V = Z'0
$$

Kết quả thu được (5, 3) x (3, 3) $\rightarrow$ (5, 3):
$$
Q =
\begin{bmatrix}
0.2449 & 0.4899 & 0.4899 \\
0.2449 & 0.4899 & 0.4899 \\
0.2449 & 0.4899 & 0.4899 \\
0.2449 & 0.4899 & 0.4899 \\
0.2449 & 0.4899 & 0.4899
\end{bmatrix},
\qquad
K = V =
\begin{bmatrix}
0.7348 & 0.7348 & 0.7348 \\
0.7348 & 0.7348 & 0.7348 \\
0.7348 & 0.7348 & 0.7348 \\
0.7348 & 0.7348 & 0.7348 \\
0.7348 & 0.7348 & 0.7348
\end{bmatrix}.
$$

In [236]:
Q = torch.matmul(z_0_layernorm, W_Q)
Q

tensor([[0.2449, 0.4899, 0.4899],
        [0.2449, 0.4899, 0.4899],
        [0.2449, 0.4899, 0.4899],
        [0.2449, 0.4899, 0.4899],
        [0.2449, 0.4899, 0.4899]])

In [237]:
K = V = torch.matmul(z_0_layernorm, W_K)       # or W_V
K, V

(tensor([[0.7348, 0.7348, 0.7348],
         [0.7348, 0.7348, 0.7348],
         [0.7348, 0.7348, 0.7348],
         [0.7348, 0.7348, 0.7348],
         [0.7348, 0.7348, 0.7348]]),
 tensor([[0.7348, 0.7348, 0.7348],
         [0.7348, 0.7348, 0.7348],
         [0.7348, 0.7348, 0.7348],
         [0.7348, 0.7348, 0.7348],
         [0.7348, 0.7348, 0.7348]]))

Lưu ý thêm, một bộ trọng số $W_Q, W_K, W_V$ sẽ sử dụng cho toàn bộ sample trong batch, áp dụng lặp lại cho mọi vector đầu vào.  
Tuy nhiên, nếu sử dụng multi-head (nhiều head) thì mỗi head sẽ có một bộ trọng số khác nhau.

### Step 03. Tính Attention Score $S$
Sau khi đã có đủ nguyên liệu $Q, K, V$, ta tiến hành tính self-attention theo công thức:
$$
S = \frac{QK^T}{\sqrt{d_k}}
$$
Trong đó $d_k$ là embedding_dim, dùng để chuẩn hóa và tránh gradient vanishing. Trong bài này thì $d_k=3$.
Với $Q$ và $K$ có shape là (num_patch, emb_dim), sau khi thực hiện phép nhân trên sẽ tạo ra ma trận $S$ có shape là (num_patch, num_patch), tức **(5, 5)**.
$$
S =
\begin{bmatrix}
0.5196 & 0.5196 & 0.5196 & 0.5196 & 0.5196 \\
0.5196 & 0.5196 & 0.5196 & 0.5196 & 0.5196 \\
0.5196 & 0.5196 & 0.5196 & 0.5196 & 0.5196 \\
0.5196 & 0.5196 & 0.5196 & 0.5196 & 0.5196 \\
0.5196 & 0.5196 & 0.5196 & 0.5196 & 0.5196
\end{bmatrix}.
$$

In [238]:
d_k = z_0.shape[1]
S = (torch.matmul(Q, K.T))/(d_k**0.5)
S

tensor([[0.5196, 0.5196, 0.5196, 0.5196, 0.5196],
        [0.5196, 0.5196, 0.5196, 0.5196, 0.5196],
        [0.5196, 0.5196, 0.5196, 0.5196, 0.5196],
        [0.5196, 0.5196, 0.5196, 0.5196, 0.5196],
        [0.5196, 0.5196, 0.5196, 0.5196, 0.5196]])

### Step 04. Tính Attention Output
Sau khi có ma trận Attention Score shape là (5, 5), Ta chuẩn hóa theo **từng hàng** các giá trị trong ma trận này bằng Softmax, sau đó nhân với ma trận giá trị $V$ để có Attention output.  
$V$ có shape là (5, 3) nên sau khi nhân sẽ trả ra output có shape là (5, 3), đúng với triết lí không làm thay đổi shape khi đi qua attention layer. 
  
Chuẩn hóa Softmax theo từng hàng, thu được ma trận $P$:
$$
P = softmax(S) = \begin{bmatrix}
0.2 & 0.2 & 0.2 & 0.2 & 0.2 \\
0.2 & 0.2 & 0.2 & 0.2 & 0.2 \\
0.2 & 0.2 & 0.2 & 0.2 & 0.2 \\
0.2 & 0.2 & 0.2 & 0.2 & 0.2 \\
0.2 & 0.2 & 0.2 & 0.2 & 0.2
\end{bmatrix}.
$$

In [239]:
import torch.nn.functional as F

P = F.softmax(S, dim=1)
P

tensor([[0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000]])

Tính Attention Output bằng phép nhân ma trận giữa $P$ (5, 5) và ma trận giá trị $V$ (5, 3), thu được Output của Layer là $A$ (5, 3):
$$
A = PV =
\begin{bmatrix}
0.7348 & 0.7348 & 0.7348 \\
0.7348 & 0.7348 & 0.7348 \\
0.7348 & 0.7348 & 0.7348 \\
0.7348 & 0.7348 & 0.7348 \\
0.7348 & 0.7348 & 0.7348
\end{bmatrix}.
$$

In [240]:
A = torch.matmul(P, V)
A

tensor([[0.7348, 0.7348, 0.7348],
        [0.7348, 0.7348, 0.7348],
        [0.7348, 0.7348, 0.7348],
        [0.7348, 0.7348, 0.7348],
        [0.7348, 0.7348, 0.7348]])

### Step 05. Skip connection
Output lúc này đang là out của Attention Layer, là $A$ có shape là (5, 3). Ta thêm thông tin từ **trước khi LayerNorm + Attention** vào $A$, tức là cộng thêm $Z_0$. Vì $Z_0$ và $A$ có cùng shape nên không cần lệnh `reshape`.  
Kết quả, ta thu được ma trận $Z_1$:
$$
Z_1 = Z_0 + A = 
\begin{bmatrix}
0.3 & 0.5 & 0.7 \\
10.6 & 12.2 & 13.8 \\
15.3 & 17.7 & 20.1 \\
28.8 & 33.6 & 38.4 \\
33.5 & 39.1 & 44.7
\end{bmatrix}
+
\begin{bmatrix}
0.7348 & 0.7348 & 0.7348 \\
0.7348 & 0.7348 & 0.7348 \\
0.7348 & 0.7348 & 0.7348 \\
0.7348 & 0.7348 & 0.7348 \\
0.7348 & 0.7348 & 0.7348
\end{bmatrix}
=
\begin{bmatrix}
1.0348 & 1.2348 & 1.4348 \\
11.3348 & 12.9348 & 14.5348 \\
16.0348 & 18.4348 & 20.8348 \\
29.5348 & 34.3348 & 39.1348 \\
34.2348 & 39.8348 & 45.4348
\end{bmatrix}
$$

In [241]:
z_1 = z_0 + A
z_1

tensor([[ 1.0348,  1.2348,  1.4348],
        [11.3348, 12.9348, 14.5348],
        [16.0348, 18.4348, 20.8348],
        [29.5348, 34.3349, 39.1348],
        [34.2348, 39.8349, 45.4348]])

### Step 06. LayerNorm lần 2
Sau khi đi qua Skip connection, vì là Pre-LayerNorm nên ta tiếp tục đưa vào Layer Norm trước khi đi qua các lớp MLP.  
Công thức sau tiếp tục được sử dụng trên từng hàng của ma trận $Z_1$:
$$
Z'_1[i, j] = \frac{Z_1[i, j] - \mu_i}{\sigma_i}
$$
Kết quả thu được ma trận $Z'_1$ như sau:
$$
Z'_1 =
\begin{bmatrix}
-1.2247 & 0 & 1.2247 \\
-1.2247 & 0 & 1.2247 \\
-1.2247 & 0 & 1.2247 \\
-1.2247 & 0 & 1.2247 \\
-1.2247 & 0 & 1.2247
\end{bmatrix}
$$

In [242]:
z_1_mean = z_1.mean(dim=1).reshape(-1, 1)
z_1_std = z_1.std(dim=1, unbiased=False).reshape(-1, 1)

z_1_layernorm = (z_1 - z_1_mean)/z_1_std
z_1_layernorm

tensor([[-1.2247,  0.0000,  1.2247],
        [-1.2247, -0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247]])

### Step 07. Feed-forward Layer.
Sử dụng cơ chế position-wise feed=forward, tức đưa qua 2 lớp MLP + 1 hàm activation function - trong đó số node ở giữa 2 layer sẽ tăng lên giúp model học được nhiều thông tin hơn (small $\rightarrow$ large $\rightarrow$ small).  
  
Trong blog này, ta không đủ không gian để thực hiện, nên chỉ feed-forward đơn giản: Đưa qua lớp Linear đầu tiên $(W_1, b_1)$, giữ nguyên shape, sau đó đưa qua activation function **GELU** và cuối cùng đưa qua lớp Linear thứ hai $(W_2, b_2)$, vẫn giữ nguyên hình dạng.
  
Bộ trọng số đầu tiên như sau:
$$
W_1 =
\begin{bmatrix}
0.1 & 0.2 & 0.3 \\
0.4 & 0.5 & 0.6 \\
0.7 & 0.8 & 0.9
\end{bmatrix},
\qquad
b_1 = [0.1,\ 0.1,\ 0.1].
$$
Hiện thực lớp Linear đầu tiên, ta được ma trận $H_1$:
$$
H_1 = Z'_1 \cdot W_1 + b_1 =
\begin{bmatrix}
0.8348 & 0.8348 & 0.8348 \\
0.8348 & 0.8348 & 0.8348 \\
0.8348 & 0.8348 & 0.8348 \\
0.8348 & 0.8348 & 0.8348 \\
0.8348 & 0.8348 & 0.8348
\end{bmatrix}.
$$

In [243]:
W_1 = torch.arange(0.1, 1.0, 0.1, dtype=torch.float32).reshape(-1, 3)
b_1 = torch.tensor([0.1]*3)
W_1, b_1

(tensor([[0.1000, 0.2000, 0.3000],
         [0.4000, 0.5000, 0.6000],
         [0.7000, 0.8000, 0.9000]]),
 tensor([0.1000, 0.1000, 0.1000]))

In [244]:
H_1 = torch.matmul(z_1_layernorm, W_1) + b_1
H_1

tensor([[0.8348, 0.8348, 0.8348],
        [0.8348, 0.8348, 0.8348],
        [0.8348, 0.8348, 0.8348],
        [0.8348, 0.8348, 0.8348],
        [0.8348, 0.8348, 0.8348]])

Kết quả lớp Linear thứ nhất là ma trận $H_1$ có shape tương đương với $Z'_1$. Tiến hành đưa qua activation function GELU được ma trận $H_2$:
$$
H_2 = GELU(H_1) =
\begin{bmatrix}
0.6662 & 0.6662 & 0.6662 \\
0.6662 & 0.6662 & 0.6662 \\
0.6662 & 0.6662 & 0.6662 \\
0.6662 & 0.6662 & 0.6662 \\
0.6662 & 0.6662 & 0.6662
\end{bmatrix}.
$$

In [245]:
H_2 = F.gelu(H_1, approximate="tanh")
H_2

tensor([[0.6662, 0.6662, 0.6662],
        [0.6662, 0.6662, 0.6662],
        [0.6662, 0.6662, 0.6662],
        [0.6662, 0.6662, 0.6662],
        [0.6662, 0.6662, 0.6662]])

Tiến hành đưa $H_2$ qua tầng tuyến tính thứ 2, bộ trọng số là:
$$
W_2 =
\begin{bmatrix}
0.2 & 0.4 & 0.6 \\
0.5 & 0.7 & 0.9 \\
0.3 & 0.5 & 0.7
\end{bmatrix},
\qquad
b_2 = [0.2,\ 0.3,\ 0.4].
$$
Kết quả thu được ma trận **output của feed-forward layer**, gọi là $Z_{FFN}$:
$$
Z_{\mathrm{FFN}} = H_2 \cdot W_2 + b_2 =
\begin{bmatrix}
0.8662 & 1.3659 & 1.8656 \\
0.8662 & 1.3659 & 1.8656 \\
0.8662 & 1.3659 & 1.8656 \\
0.8662 & 1.3659 & 1.8656 \\
0.8662 & 1.3659 & 1.8656
\end{bmatrix}.
$$

In [246]:
W_2 = torch.tensor([
    [0.2, 0.4, 0.6],
    [0.5, 0.7, 0.9],
    [0.3, 0.5, 0.7]
], dtype=torch.float32)
b_2 = torch.tensor([0.2, 0.3, 0.4], dtype=torch.float32)
W_2, b_2

(tensor([[0.2000, 0.4000, 0.6000],
         [0.5000, 0.7000, 0.9000],
         [0.3000, 0.5000, 0.7000]]),
 tensor([0.2000, 0.3000, 0.4000]))

In [247]:
Z_FFN = torch.matmul(H_2, W_2) + b_2
Z_FFN

tensor([[0.8662, 1.3659, 1.8656],
        [0.8662, 1.3659, 1.8656],
        [0.8662, 1.3659, 1.8656],
        [0.8662, 1.3659, 1.8656],
        [0.8662, 1.3659, 1.8656]])

### Step 08. Skip connection lần 2.
Tiếp tục cộng ma trận $z_1$ **trước khi LayerNorm** vào ma trận $Z_FFN$ hiện tại để thêm thông tin. Shape tương đương nên không cần `reshape`.
$$
Z_2 = Z_1 + Z_{\mathrm{FFN}} =
\begin{bmatrix}
1.9010 & 2.6008 & 3.3005 \\
12.2010 & 14.3008 & 16.4005 \\
16.9010 & 19.8008 & 22.7005 \\
30.4010 & 35.7008 & 41.0005 \\
35.1010 & 41.2008 & 47.3005
\end{bmatrix}.
$$

In [248]:
z_2 = z_1 + Z_FFN
z_2

tensor([[ 1.9010,  2.6008,  3.3005],
        [12.2010, 14.3008, 16.4005],
        [16.9010, 19.8008, 22.7005],
        [30.4010, 35.7008, 41.0005],
        [35.1010, 41.2008, 47.3005]])

Đầu ra của Skip connection cũng chính là đầu ra của một khối Transformer Encoder, có shape là (5, 3) giống với input, đúng như kì vọng.  
Đầu ra (5, 3), bao gồm 1 [CLS] Token và 4 patch, mỗi token/patch được mã hóa thành 3 con số (emb_dim=3).
### Trích xuất [CLS] Token
[CLS] Token chính là hàng đầu tiên trong output của Attention Layer:
$$
Z_{CLS} = row_1(Z_2) = [1.9010, 2.6008, 3.3005] \in \mathbb{R}^{1 \times 3}
$$

In [249]:
torch.set_printoptions(profile="default")
Z_CLS = z_2[0].reshape(1, -1)
Z_CLS, Z_CLS.shape

(tensor([[1.9010, 2.6008, 3.3005]]), torch.Size([1, 3]))

[CLS] Token đóng vai trò học ngữ nghĩa cho **toàn bộ ảnh** thay cho các patch, sẽ được sử dụng để đưa vào MLP và phân loại.

### Classification Head
[CLS] Token sau khi được lấy ra sẽ đưa vào một mạng classifier, bao gồm 2 Tầng tuyến tính và 1 activation Function (GELU) để tính xác suất & phân loại.
Với bộ trọng số đầu tiên:
$$
W_{\mathrm{hidden}} =
\begin{bmatrix}
0.2 & 0.3 & 0.4 \\
0.5 & 0.6 & 0.7 \\
0.8 & 0.9 & 1.0
\end{bmatrix},
\qquad
b_{\mathrm{hidden}} = [0.1,\ 0.1,\ 0.1].
$$
Ta kết hợp Linear + GELU trong một phép tính, được ma trận hidden H_CLS:
$$
H_{CLS} = GELU(Z_{CLS} \cdot W_{hidden} + b_{hidden}) 
$$
$Z_{CLS}$ có shape là (1, 3), $W_{hidden}$ có shape là (3, 3) $\rightarrow$ $H_{CLS}$ có shape là (1, 3), là một vector ẩn được đưa vào tầng linear tiếp theo. 
$$
H = [\,4.4209\;\; 5.2012 \;\; 5.9814\,].
$$

In [250]:
W_hidden = torch.arange(0.2, 1.1, 0.1, dtype=torch.float32).reshape(-1, 3)
b_hidden = torch.tensor([0.1]*3)

W_hidden, b_hidden

(tensor([[0.2000, 0.3000, 0.4000],
         [0.5000, 0.6000, 0.7000],
         [0.8000, 0.9000, 1.0000]]),
 tensor([0.1000, 0.1000, 0.1000]))

In [251]:
H_CLS = F.gelu(torch.matmul(Z_CLS, W_hidden) + b_hidden)
H_CLS

tensor([[4.4209, 5.2012, 5.9814]])

Ở tầng Linear Thứ 2, ta có bộ trọng số khác:
$$
W_{\mathrm{out}} =
\begin{bmatrix}
0.3 & 0.4 \\
0.5 & 0.6 \\
0.7 & 0.8
\end{bmatrix},
\qquad
b_{\mathrm{out}} = [0.1,\ 0.2].
$$
Kết quả của phép tuyến tính gọi là $O_{logits}$, thể hiện việc ánh xạ từ không gian ẩn sang không gian nhãn (từ 3 emb thành 2 emb, ma trận trọng số là (3, 2)):
$$
O_{logits} = H_{CLS} \cdot W_{out} + b_{out} = [ 8.2139 \;\; 9.8742\,]
$$

In [252]:
W_out = torch.linspace(0.3, 0.8, steps=6, dtype=torch.float32).reshape(-1, 2)
b_out = torch.tensor([0.1, 0.2])
W_out, b_out

(tensor([[0.3000, 0.4000],
         [0.5000, 0.6000],
         [0.7000, 0.8000]]),
 tensor([0.1000, 0.2000]))

In [253]:
o_logits = torch.matmul(H_CLS, W_out) + b_out
o_logits

tensor([[8.2139, 9.8742]])

### Dùng Softmax để chuyển thành xác suất
Giả sử có 2 lớp. Ta chuyển thành xác suất rồi dự đoán dựa theo xác suất, hoặc train thêm nhiều epochs khác.
$$
P_{\mathrm{output}, i}
=
\frac{\exp\!\left(O_{\mathrm{logits}, i}\right)}
{\sum_{j} \exp\!\left(O_{\mathrm{logits}, j}\right)}
= \left[\,0.1597 \;\; 0.8403\,\right]
$$

In [255]:
P_output = F.softmax(o_logits, dim=1)
P_output

tensor([[0.1597, 0.8403]])

### Kết thúc -- Vision Transformer.